# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

FlyRank publishes content for many clients and cannot review every page by hand. The practical question is simple: **which already-visible pages should an editor review first for a refresh?** This is a prioritization decision, not a prediction of Google's algorithm. One row is one content page, the output is a ranked queue with a reason code and a suggested action, and the person acting is an editor who can work maybe fifty pages a month. A wrong call has a real cost: a rewrite on a page that did not need it wastes editor time and can lose rankings the page already had, while a page that quietly declined and never surfaced in the queue keeps bleeding traffic. Ranking is worth automating because the signals are many and tangled and no single hand rule orders them well, but the final call stays with a person.

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
visible = df["impressions_90d"] >= 100

print("total pages:", len(df))
print("visible (impressions_90d >= 100):", int(visible.sum()))
print("below threshold:", int((~visible).sum()))
print("visible base rate declining:", round(df.loc[visible, "is_declining"].mean(), 3))
print("clients:", df["client_id"].nunique())

total pages: 30000
visible (impressions_90d >= 100): 22006
below threshold: 7994
visible base rate declining: 0.598
clients: 32


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The model, baseline, and validation run on the 30,000-row anonymized starter slice that ships in this repo: one row per content page, 32 pseudonymized clients, metrics over a trailing 90-day window. Keeping the model on the committed slice means anyone can rerun it without gated access. My data contract (`w03_data_contract.ipynb`) verified the same lane on the full warehouse release, the March 2026 partition, roughly 9.8 million page-days, to confirm the grain and windows hold at scale.

What I excluded, and why, matters more than what I kept. The label comes from `trend_direction`, so `trend_direction`, `trend_pct`, and the derived `is_declining` never enter the features. The 90-day and last-30-day windows overlap the label's own window, so they are excluded as features too; only the earlier `*_prev_30d` window and static page properties are fair game. Pseudonymous IDs group and split the data, never train it. Nothing client-identifying appears anywhere in the repo.

In [2]:
feature_num = ["content_age_days", "days_since_last_update", "word_count", "char_count",
               "search_volume", "competition", "cpc",
               "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
excluded = ["trend_direction", "trend_pct", "is_declining", "impressions_90d",
            "impressions_last_30d", "ctr", "avg_position", "content_id", "client_id"]

df["has_keyword"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
for c in ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]:
    df["log_" + c] = np.log1p(df[c].fillna(0))
cats = pd.get_dummies(df[["content_type", "main_intent", "competition_level"]].fillna("unknown"), drop_first=True)
extra = ["has_keyword", "has_word_count", "log_impressions_prev_30d", "log_clicks_prev_30d", "log_sessions_prev_30d"]
X = pd.concat([df[feature_num].fillna(0), df[extra], cats], axis=1)
y = df["is_declining"].values
groups = df["client_id"].values
vmask = visible.values

print("features used:", X.shape[1])
print("excluded on purpose:", excluded)

features used: 24
excluded on purpose: ['trend_direction', 'trend_pct', 'is_declining', 'impressions_90d', 'impressions_last_30d', 'ctr', 'avg_position', 'content_id', 'client_id']


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

The label is a proxy: `is_declining = (trend_direction == "down")`, a defined stand-in for "worth a refresh slot", not a measured business outcome. The baseline is the transparent CTR-fix rule from Week 4, scored here as a decline ranker on the same folds, plus a naive staleness ranker as a floor. The model is Logistic Regression over the leakage-safe features, chosen after an exhaustive search under `work/experiments/` compared twelve model families; the trees and boosters ranked worse at the top of the queue, and the only durable gain was widening the training population to include sub-threshold pages while still scoring only visible ones.

Validation is client-grouped 5-fold: no client sits in both train and test, because a random split lets a model memorize a client and report skill it does not have. Every number below is out-of-fold on unseen clients, with the base rate beside it.

In [3]:
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def logreg():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
def p_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

gp = df[(df["avg_position"] > 0) & (df["position_tier"].isin(["top_3", "page_1", "striking"])) & visible]
exp_ctr = gp.groupby("position_tier")["ctr"].median()
gap = df["position_tier"].map(exp_ctr) - df["ctr"]
ctrfix = np.where((df["avg_position"] > 0) & (df["position_tier"].isin(["top_3", "page_1", "striking"])) & (gap > 0),
                  df["impressions_90d"] * gap, 0.0)
staleness = df["days_since_last_update"].values

print("label: is_declining = (trend_direction == 'down'); baseline: Week-4 CTR-fix + staleness floor")
print("validation: client-grouped 5-fold; metric: ROC-AUC and Precision@50")

label: is_declining = (trend_direction == 'down'); baseline: Week-4 CTR-fix + staleness floor
validation: client-grouped 5-fold; metric: ROC-AUC and Precision@50


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Same visible pages, same client-grouped folds, same metrics. The model beats both baselines, and the widened-training variant edges the visible-only model at the top of the queue. The lift over the CTR-fix baseline is real but modest, roughly 1.4x on Precision@50, not the 3x a weak baseline would manufacture. That gap is the point: a fair baseline keeps the model honest.

In [4]:
rows = []
folds = list(GroupKFold(n_splits=5).split(X, y, groups))
for name in ["baseline: CTR-fix", "baseline: staleness", "model: LogReg (visible-train)", "model: LogReg (widened-train)"]:
    aucs, pks = [], []
    for tr, te in folds:
        vte = te[vmask[te]]
        vtr = tr[vmask[tr]]
        if name == "baseline: CTR-fix": s = ctrfix[vte]
        elif name == "baseline: staleness": s = staleness[vte]
        elif name == "model: LogReg (visible-train)":
            m = logreg().fit(X.iloc[vtr], y[vtr]); s = m.predict_proba(X.iloc[vte])[:, 1]
        else:
            m = logreg().fit(X.iloc[tr], y[tr]); s = m.predict_proba(X.iloc[vte])[:, 1]
        aucs.append(roc_auc_score(y[vte], s)); pks.append(p_at_k(s, y[vte]))
    rows.append({"method": name, "ROC_AUC": round(np.mean(aucs), 3), "P@50": round(np.mean(pks), 3)})

results = pd.DataFrame(rows)
base_rate = round(df.loc[visible, "is_declining"].mean(), 3)
print("visible base rate:", base_rate)
print(results.to_string(index=False))
results.to_json("../outputs/capstone_results.json", orient="records", indent=2)

visible base rate: 0.598
                       method  ROC_AUC  P@50
            baseline: CTR-fix    0.568 0.600
          baseline: staleness    0.482 0.664
model: LogReg (visible-train)    0.607 0.740
model: LogReg (widened-train)    0.637 0.808


## 5. Limitations

*What this work cannot claim.*

Three honest limits. First, **leakage is easy and I refused it**: the cell below shows that adding the label's own time window drives AUC toward a fake 0.9+, which is why the honest features stay near 0.6. Second, **a random split flatters the model**: the same pipeline scores higher when clients leak across the split than on held-out clients, and that gap is memorization, not skill. Third, **decision value is not proven**: the model ranks declining pages above the base rate, but whether refreshing them recovers traffic is an intervention question that needs an A/B test this snapshot cannot run. The widening gain also carries a wide bootstrap interval, so it is directional, not measured. This is decision-support for an editor's queue, nothing more.

In [5]:
tr0, te0 = folds[0]
vte0, vtr0 = te0[vmask[te0]], tr0[vmask[tr0]]
def auc_with(cols):
    XX = pd.concat([X, df[cols].fillna(0)], axis=1) if cols else X
    m = logreg().fit(XX.iloc[vtr0], y[vtr0]); return round(roc_auc_score(y[vte0], m.predict_proba(XX.iloc[vte0])[:, 1]), 3)

print("Leakage escalation (grouped fold, adding banned columns):")
print("  honest features            ", auc_with([]))
print("  + ctr, avg_position        ", auc_with(["ctr", "avg_position"]))
print("  + impressions_90d          ", auc_with(["ctr", "avg_position", "impressions_90d"]))
print("  + impressions_last_30d     ", auc_with(["ctr", "avg_position", "impressions_90d", "impressions_last_30d"]))
print("  + trend_pct (the label)    ", auc_with(["trend_pct"]))

grp = np.mean([roc_auc_score(y[te[vmask[te]]], logreg().fit(X.iloc[tr[vmask[tr]]], y[tr[vmask[tr]]]).predict_proba(X.iloc[te[vmask[te]]])[:, 1]) for tr, te in folds])
Xv, yv = X[vmask].reset_index(drop=True), y[vmask]
rnd = np.mean([roc_auc_score(yv[te], logreg().fit(Xv.iloc[tr], yv[tr]).predict_proba(Xv.iloc[te])[:, 1]) for tr, te in KFold(5, shuffle=True, random_state=42).split(Xv)])
print(f"\nRandom 5-fold AUC {rnd:.3f} vs client-grouped {grp:.3f}  -> inflation {rnd-grp:.3f} (memorization)")

Leakage escalation (grouped fold, adding banned columns):
  honest features             0.627
  + ctr, avg_position         0.662
  + impressions_90d           0.698


  + impressions_last_30d      0.949
  + trend_pct (the label)     0.997



Random 5-fold AUC 0.685 vs client-grouped 0.607  -> inflation 0.078 (memorization)


## 6. Ranked recommendations

*The action playbook output: the paper's recommendations section.*

The queue turns scores into moves an editor can make tomorrow. Each of the top fifty pages gets one action and one reason code. The mix leans on refreshing content and rewriting titles, with a smaller share for relevance and internal-link work. Two archetypes dominate: maturing assets that are slipping, and neglected earners that pull traffic but get no attention. The reason codes are honest about coverage: for nearly half of visible pages the rule finds no clear lever, and the notebook says so rather than inventing one.

In [6]:
import json as _json
pb = _json.load(open("../outputs/playbook_metrics.json"))
print("Top-50 queue action mix:")
for a, n in pb["queue"]["action_mix"].items(): print(f"  {n:>3}  {a}")
print("\nReason-code share across all visible pages:")
for r, s in pb["reason_code_share_all_visible"].items(): print(f"  {s:>5.1%}  {r}")
print("\nDecline share by content age tier (older pages are steadier, not stalest):")
for t, d in pb["decay_by_age_tier"].items(): print(f"  {t:>8}  {d['share_declining']:.3f}  (n={d['pages']})")

Top-50 queue action mix:
   19  refresh the content
   15  rewrite title and meta
   14  relevance work and internal links
    1  watch only
    1  expand the page

Reason-code share across all visible pages:
  48.2%  no_clear_lever
  25.3%  low_visibility
  12.3%  stale_but_earning
  12.2%  low_ctr_for_position
   2.0%  thin_for_demand

Decline share by content age tier (older pages are steadier, not stalest):
     31-90  0.681  (n=304)
    91-180  0.692  (n=8717)
   181-365  0.606  (n=7650)
      365+  0.427  (n=5335)


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Two charts carry the story on the deployed page: how decline varies with page age and freshness, and the action mix of the top-50 queue. Both are regenerated here so the page and the notebook never drift apart.

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, os
os.makedirs("../figures", exist_ok=True)

age = pb["decay_by_age_tier"]; fresh = pb["freshness_by_tier"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(list(age.keys()), [d["share_declining"] for d in age.values()], color="#4C78A8")
ax[0].axhline(base_rate, ls="--", color="#666"); ax[0].set_title("Decline share by content age"); ax[0].set_ylabel("share declining")
ax[1].bar(list(fresh.keys()), [d["share_declining"] for d in fresh.values()], color="#72B7B2")
ax[1].axhline(base_rate, ls="--", color="#666"); ax[1].set_title("Decline share by days since update")
for a in ax: a.set_ylim(0, 0.8); a.tick_params(axis="x", rotation=20)
fig.tight_layout(); fig.savefig("../figures/decline_by_age_and_freshness.png", dpi=110); plt.close(fig)

mix = pb["queue"]["action_mix"]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(list(mix.keys())[::-1], list(mix.values())[::-1], color="#E45756")
ax.set_title("Top-50 refresh queue: action mix"); ax.set_xlabel("pages")
fig.tight_layout(); fig.savefig("../figures/queue_action_mix.png", dpi=110); plt.close(fig)
print("wrote ../figures/decline_by_age_and_freshness.png and ../figures/queue_action_mix.png")

wrote ../figures/decline_by_age_and_freshness.png and ../figures/queue_action_mix.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.

## 8. Five-minute demo outline

A showcase walkthrough, timed for five minutes.

- **Question (0:30).** FlyRank editors own hundreds of live pages and can review about fifty a month. Which already-visible pages should they refresh first?
- **Method (1:30).** Rank pages by how likely they are to be declining, using only leakage-safe signals from the prior window. Validate on client-grouped folds so no client leaks between train and test. Compare against a fair Week-4 rule baseline on the same split and metric.
- **One chart (1:00).** Decline share by page age: pages three to six months old slip most (0.69), and pages past a year are the steadiest (0.43). Age is a real signal, and it is not the naive "older is stale" story.
- **One honest result (1:00).** Logistic Regression reaches Precision@50 of 0.81 against a 0.60 base rate, a modest 1.35x lift over the baseline, not 3x. If I let the label's own time window into the features, AUC jumps to 0.95, which is exactly the cheat I refused.
- **One recommendation (1:00).** Work the top-50 queue: 19 refreshes, 15 title rewrites, 14 relevance fixes. Skip the 48% of pages where the rule finds no clear lever, and treat the whole thing as decision-support, not proof a refresh pays off.

## 9. Shareable cuts

**Social post (methodology).**
I spent an internship learning that the hard part of an ML project is not the model, it is refusing the easy win. On real search data I built a page-refresh ranker and could have hit 0.95 AUC in one line by feeding it the label's own time window. The honest number, on features known before the outcome and validated on clients the model never saw, is closer to 0.64. The gap between those two is the whole job. Full write-up and repo in the comments.

**Employer summary (three sentences).**
I built a content-refresh prioritization model on the FlyRank internship dataset (30,000 pages across 32 clients, anonymized production search data), turning it into a ranked queue an editor can act on with a reason code per page. It beats a fair transparent baseline on client-grouped validation (Precision@50 0.81 versus 0.60) while staying leakage-safe, and I documented where a naive setup would have inflated the result to a fake 0.95. The deliverable is a deployed research paper with honest, decision-support framing, reproducible end to end from the repo.